# PCA on the Iris dataset

The Iris dataset is a classical dataset to learn to use Machine Learning methods. We're going to use it here to see how to use PCA and some traps to avoid.

## Getting data

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy
import plotly.express as px
import scipy.stats
import sklearn.datasets
import sklearn.decomposition
import sklearn.preprocessing


df = px.data.iris()

print("5 first lines")
print(df.head(5))
print()
print("Description of the numerical features")
print(df.describe())

## Preparing features and targets

The targets will be used here for visualization only, not for training.

In [ ]:
X = df.drop(columns=["species", "species_id"]).values
y = df["species_id"].values

## Visualizing the data

In [ ]:
styling = dict(template="seaborn", width=800, height=600)

def show(fig):
  fig.update_traces(marker=dict(line=dict(width=1,
                                          color='DarkSlateGrey')),
                    selector=dict(mode='markers'))
  fig.show()


fig = px.scatter_3d(df,
                    x="sepal_length",
                    y="sepal_width",
                    z="petal_length",
                    color="species",
                    hover_data=["petal_width"], # Overlay petal_width
                    **styling)
show(fig)

## Computing PCA

In [ ]:
pca = sklearn.decomposition.PCA(n_components=3)
pca.fit(X)

X_pca = pca.transform(X)

print("Explained variance:", pca.explained_variance_ratio_)
print("Cumulated explained variance:",
      numpy.cumsum(pca.explained_variance_ratio_))

## Visualizing the learned linear projections

In [ ]:
fig = px.scatter_3d(X_pca, x=0, y=1, z=2, color=df["species"], **styling)
show(fig)

## Dimensionality reduction

In [ ]:
fig = px.scatter(X_pca, x=0, y=1, color=df["species"], **styling)
show(fig)

## Principal components exploration

In [ ]:
print("Components", pca.components_)

for k, v in enumerate(numpy.argsort(numpy.abs(pca.components_))):
  print()
  print(f"Features coefficients for component {k}")
  for i in v[::-1]:
    print(f"  {df.columns[i]}: {pca.components_[k][i]}")

# Features units and variance

Those results should be interpreted with caution: we could be lead to believe that 92% of the variance is explained thanks to the first component. However, we did not standardize the data, so a feature with bigger values will likely dominate variance.

There is a simple solution: standardizing.

In [ ]:
X_scaler = sklearn.preprocessing.StandardScaler()
X_scaler.fit(X)
X_scaled = X_scaler.transform(X)
#X_scaled = X_scaler.fit_transform(X)
print(X_scaled.shape)

In [ ]:
pca = sklearn.decomposition.PCA(n_components=3)
pca.fit(X_scaled)
X_scaled_pca = pca.transform(X_scaled)

print("Explained variance for each component:",
      ", ".join(f"{v:.2f}" for v in pca.explained_variance_ratio_))
print("Cumulated explained variance:",
      ", ".join(f"{v:.2f}"
      for v in numpy.cumsum(pca.explained_variance_ratio_)))

In [ ]:
fig = px.scatter_3d(X_scaled_pca, x=0, y=1, z=2, color=df["species"], **styling)
show(fig)

In [ ]:
print("Components", pca.components_)

for k, v in enumerate(numpy.argsort(numpy.abs(pca.components_))):
  print()
  print(f"Features coefficients for component {k}")
  for i in v[::-1]:
    print(f"  {df.columns[i]}: {pca.components_[k][i]}")

In [ ]:
t = numpy.linspace(0, numpy.pi * 2, 100)

plt.figure(figsize=(10, 10))
plt.xlim((-1.1, 1.1))
plt.ylim((-1.1, 1.1))
for feature in range(X_scaled.shape[1]):
  corr_0 = scipy.stats.pearsonr(X_scaled[:, feature], X_scaled_pca[:, 0])[0]
  corr_1 = scipy.stats.pearsonr(X_scaled[:, feature], X_scaled_pca[:, 1])[0]
  r = math.atan2(corr_1, corr_0)
  plt.arrow(0, 0, corr_0, corr_1, length_includes_head=True, head_width=0.03)
  plt.text(corr_0 / 2 + (math.cos(r + 0.1) - math.cos(r)) / 2,
           corr_1 / 2 + (math.sin(r + 0.1) - math.sin(r)) / 2,
           df.columns[feature],
           rotation=numpy.degrees(r),
           size=16,
           ha="center",
           va="center")
plt.xlabel("First component")
plt.ylabel("Second component")
plt.plot(numpy.cos(t), numpy.sin(t))
plt.show()